# Learning curves from training history

**Kernel:** `conda env:.conda-diffusion`

Looks for `progress.csv` under each config's `OPENAI_LOGDIR`:
`/scratch/7DayExclusive/munjung/anomaly-detection/training/diffusion/<config>/`

**Important:** older runs may have written everything into a shared
`train/diffusion-anomaly/results/` because `logger.configure()` ignored
`OPENAI_LOGDIR`. That is fixed now — **restart training** so linear and
anisotropic no longer collide in the same folder.

Figures → `/exp/sbnd/data/users/munjung/anomaly-detection/training/curves/`.


In [ ]:
from __future__ import annotations

import csv
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

APP_ROOT = Path("/exp/sbnd/app/users/munjung/anomaly-detection")
sys.path.insert(0, str(APP_ROOT))
from configs.paths import DATA_ROOT, SCRATCH_TRAINING, ensure_layout
from configs.train_configs import TRAIN_CONFIGS

ensure_layout()
CURVES = DATA_ROOT / "training" / "curves"
CURVES.mkdir(parents=True, exist_ok=True)

RUN_ROOT = SCRATCH_TRAINING / "diffusion"
# Shared fallback used by the pre-fix logger bug (both jobs wrote here).
LEGACY_RESULTS = APP_ROOT / "train" / "diffusion-anomaly" / "results"

CONFIGS = [c.name for c in TRAIN_CONFIGS]


def resolve_progress(name: str) -> Path | None:
    candidates = [
        RUN_ROOT / name / "progress.csv",
        DATA_ROOT / "training" / "diffusion" / name / "progress.csv",
    ]
    for p in candidates:
        if p.is_file():
            return p
    return None


print("RUN_ROOT", RUN_ROOT, "exists=", RUN_ROOT.exists())
print("LEGACY_RESULTS", LEGACY_RESULTS, "progress=", (LEGACY_RESULTS / "progress.csv").is_file())
for name in CONFIGS:
    prog = resolve_progress(name)
    d = RUN_ROOT / name
    print(f"  {name}: progress={prog}  scratch_dir_exists={d.exists()}")


In [ ]:
def load_progress(path: Path):
    with path.open() as f:
        rows = list(csv.DictReader(f))
    return rows


fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plotted = 0
used_legacy = False

for name in CONFIGS:
    prog = resolve_progress(name)
    label = name
    if prog is None and name in ("linear", "anisotropic") and (LEGACY_RESULTS / "progress.csv").is_file():
        # Only show the shared file once — it is a mix/overwrite of colliding runs.
        if used_legacy:
            continue
        prog = LEGACY_RESULTS / "progress.csv"
        label = "LEGACY shared results/ (pre-OPENAI_LOGDIR fix)"
        used_legacy = True
        print("WARNING: plotting shared", prog, "- restart training so configs get separate log dirs")
    if prog is None or not prog.is_file():
        continue
    rows = load_progress(prog)
    if not rows:
        continue
    step_key = next(k for k in ("step", "steps", "iteration") if k in rows[0])
    loss_key = next((k for k in ("loss", "mse", "train_loss") if k in rows[0]), None)
    if loss_key is None:
        loss_key = next(k for k in rows[0] if "loss" in k.lower())
    steps = np.array([float(r[step_key]) for r in rows if r.get(step_key) not in (None, "")])
    loss = np.array([float(r[loss_key]) for r in rows if r.get(step_key) not in (None, "") and r.get(loss_key) not in (None, "")])
    n = min(len(steps), len(loss))
    steps, loss = steps[:n], loss[:n]
    axes[0].plot(steps, loss, label=label, alpha=0.85)
    if len(loss) > 20:
        w = max(5, len(loss) // 50)
        kernel = np.ones(w) / w
        smooth = np.convolve(loss, kernel, mode="valid")
        axes[1].plot(steps[: len(smooth)], smooth, label=label)
    else:
        axes[1].plot(steps, loss, label=label)
    plotted += 1
    print(f"{label}: {len(steps)} rows from {prog}")

axes[0].set_title("raw loss"); axes[0].set_xlabel("step"); axes[0].legend()
axes[1].set_title("smoothed"); axes[1].set_xlabel("step"); axes[1].legend()
out = CURVES / "all_configs_curves.pdf"
if plotted:
    fig.savefig(out, bbox_inches="tight")
    print("saved", out)
else:
    print("No progress.csv found yet — train with fixed logger, or check LEGACY_RESULTS")
plt.show()
